In [1]:
import pandas as pd

df = pd.read_csv("/home/jovyan/work/data/creditcardfraud/creditcard.csv")
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [2]:
# Basic info
print("Shape:", df.shape)
print("\nFraud rate:")
print(df["Class"].value_counts(normalize=True))

Shape: (284807, 31)

Fraud rate:
Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64


In [3]:
# Missing values
df.isnull().sum().sum()

0

In [4]:
# V features correlation with fraud
v_cols = [col for col in df.columns if col.startswith('V')]
fraud_corr = df[v_cols + ['Class']].corr()['Class'].sort_values(ascending=False)
print("Top 10 positive correlations:")
print(fraud_corr.head(11))  # 11 because Class itself is 1.0
print("\nTop 10 negative correlations:")
print(fraud_corr.tail(10))

Top 10 positive correlations:
Class    1.000000
V11      0.154876
V4       0.133447
V2       0.091289
V21      0.040413
V19      0.034783
V20      0.020090
V8       0.019875
V27      0.017580
V28      0.009536
V26      0.004455
Name: Class, dtype: float64

Top 10 negative correlations:
V9    -0.097733
V1    -0.101347
V18   -0.111485
V7    -0.187257
V3    -0.192961
V16   -0.196539
V10   -0.216883
V12   -0.260593
V14   -0.302544
V17   -0.326481
Name: Class, dtype: float64


In [5]:
# Amount distribution
print("Legitimate transactions:")
print(df[df["Class"]==0]["Amount"].describe())
print("\nFraudulent transactions:")
print(df[df["Class"]==1]["Amount"].describe())

Legitimate transactions:
count    284315.000000
mean         88.291022
std         250.105092
min           0.000000
25%           5.650000
50%          22.000000
75%          77.050000
max       25691.160000
Name: Amount, dtype: float64

Fraudulent transactions:
count     492.000000
mean      122.211321
std       256.683288
min         0.000000
25%         1.000000
50%         9.250000
75%       105.890000
max      2125.870000
Name: Amount, dtype: float64


In [6]:
# Time is in seconds from first transaction
# Convert to hours
df['hour'] = (df['Time'] / 3600) % 24

# Fraud rate by hour
print(df.groupby('hour')['Class'].mean().sort_values(ascending=False).head(10))

hour
2.080000     1.0
3.860556     1.0
11.859722    1.0
4.492222     1.0
3.646111     1.0
5.680833     1.0
0.649167     1.0
7.979444     1.0
4.506111     1.0
6.579722     1.0
Name: Class, dtype: float64


In [7]:
# Better - only hours with at least 100 transactions
hourly = df.groupby('hour').agg({'Class': ['sum', 'count', 'mean']})
hourly = hourly[hourly[('Class', 'count')] >= 100]
hourly_sorted = hourly.sort_values(('Class', 'mean'), ascending=False)
print(hourly_sorted.head(15))

Empty DataFrame
Columns: [(Class, sum), (Class, count), (Class, mean)]
Index: []


In [8]:
# Round to whole hours
df['hour_rounded'] = (df['Time'] / 3600).astype(int) % 24

# Fraud rate by hour
hourly = df.groupby('hour_rounded')['Class'].agg(['sum', 'count', 'mean']).sort_values('mean', ascending=False)
print(hourly)

              sum  count      mean
hour_rounded                      
2              57   3328  0.017127
4              23   2209  0.010412
3              17   3492  0.004868
5              11   2990  0.003679
7              23   7243  0.003175
11             53  16856  0.003144
1              10   4220  0.002370
6               9   4101  0.002195
18             33  17039  0.001937
23             21  10938  0.001920
17             29  16166  0.001794
15             26  16461  0.001579
14             23  16570  0.001388
16             22  16453  0.001337
19             19  15649  0.001214
13             17  15365  0.001106
12             17  15420  0.001102
20             18  16756  0.001074
9              16  15838  0.001010
21             16  17703  0.000904
8               9  10276  0.000876
0               6   7695  0.000780
22              9  15441  0.000583
10              8  16598  0.000482
